In [ ]:
!pip install -q kaggle
!pip install torchinfo

import math
import copy
import torch
import inspect
import pandas as pd
import torch.nn as nn

from tqdm import tqdm
from torchinfo import summary
from google.colab import files
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoConfig, DataCollatorWithPadding, AutoTokenizer

files.upload()

!kaggle datasets download datascikhan/ai-vs-human-text-classification-benchmark-50k
!unzip /content/ai-vs-human-text-classification-benchmark-50k.zip

path = "/content/ai-vs-human-text-classification-benchmark-50k.zip"
df = pd.read_csv(path)
df.head()

Saving kaggle (1).json to kaggle (1) (1).json
Dataset URL: https://www.kaggle.com/datasets/datascikhan/ai-vs-human-text-classification-benchmark-50k
License(s): CC0-1.0
ai-vs-human-text-classification-benchmark-50k.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  /content/ai-vs-human-text-classification-benchmark-50k.zip
replace AI_vs_Human_Text_Classification_Benchmark.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: AI_vs_Human_Text_Classification_Benchmark.csv  


,Text_ID,Raw_Text,Label,Source_Model,Model_Size,Model_Temperature,Generation_Prompt_Type,RAG_Context_Provided,Detection_Score_Benchmark_Model_A,Word_Count,...,Stopword_Density,Flesch_Kincaid_Grade_Level,Smog_Index,Average_Syllables_Per_Word,Contains_Code_Snippet,Capital_Letter_Count,Formal_Tone_Score,Emotion_Tone_Intensity,Coherence_Score,Burstiness_Score
0,TXT_09927,History has always been a subject of intense d...,0,Human_Origin,Human,NaN,Human_Writing,Human,21.510198,219,...,0.438356,11.785556,12.522536,2.027397,0,28,100.000000,16.518485,81.544039,48.832741
1,TXT_35284,"When it comes Similarly, Philosophy, there are...",0,Human_Origin,Human,NaN,Human_Writing,Human,78.303093,293,...,0.457338,11.612546,11.958738,2.000000,0,37,100.000000,23.965492,93.325738,65.352863
2,TXT_27383,Let me tell you about Programming. It's one of...,0,Human_Origin,Human,NaN,Human_Writing,Human,83.110976,161,...,0.496894,11.031269,12.423615,1.850932,0,20,91.519314,31.751173,57.563496,58.196010
3,TXT_36542,Have you ever considered the role of Travel in...,0,Human_Origin,Human,NaN,Human_Writing,Human,42.621955,106,...,0.500000,9.075098,11.429528,1.650943,0,11,54.611734,40.980060,42.889109,68.661004
4,TXT_23205,"In my experience with Psychology, I've noticed...",0,Human_Origin,Human,NaN,Human_Writing,Human,100.000000,76,...,0.486842,13.749167,15.247665,1.842105,0,6,50.752909,43.610382,28.351032,44.903004


In [ ]:
exp = pd.read_csv(path)
exp = exp[['Raw_Text', 'Label']]
print(exp['Raw_Text'].str.len().max())
print("--- Label 비율 ---")
print(exp['Label'].value_counts(normalize=True))

7624
--- Label 비율 ---
Label
0    0.5
1    0.5
Name: proportion, dtype: float64


In [ ]:
MODEL_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

df = pd.read_csv(path)
df = df[['Raw_Text', 'Label']]
df.columns = ['text', 'label']

df = Dataset.from_pandas(df)
df = df.class_encode_column('label')

dataset = df.train_test_split(
    seed = 42,
    test_size = 2000,
    train_size = 8000,
    stratify_by_column = 'label',
)

def preprocess(examples):
  encoding = tokenizer(
      examples['text'],
      truncation = True,
      max_length = 256,
  )
  return encoding

tokenized_dataset = dataset.map(
    preprocess,
    batched = True,
    batch_size = 5000,
    remove_columns = dataset['train'].column_names[0]
)

tokenized_dataset.set_format("torch")

Stringifying the column:   0%|          | 0/50000 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True
)

train_dataloader = DataLoader(
    tokenized_dataset['train'],
    batch_size=64,
    shuffle=True,
    collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_dataset['test'],
    batch_size=64,
    shuffle=False,
    collate_fn=data_collator
)

In [ ]:
class CustomModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.bert = AutoModel.from_pretrained('bert-base-uncased')
    self.classifier = nn.Linear(self.bert.config.hidden_size, 2)

  def forward(self, input_ids, attention_mask = None, token_type_ids = None, **kwargs):
    outputs = self.bert(
        input_ids = input_ids,
        attention_mask = attention_mask,
        token_type_ids = token_type_ids
        )
    logits = self.classifier(outputs.pooler_output)
    return logits

model = CustomModel()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import torch
import torch.nn as nn
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

epochs = 1
learning_rate = 2e-5

optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

criterion = nn.CrossEntropyLoss()

In [ ]:
# 3. Epoch 단위 학습 및 평가 루프
for epoch in range(epochs):
    print(f"\n======== Epoch {epoch + 1} / {epochs} ========")

    # ---------------------------------------------------------
    # [1] Training Loop
    # ---------------------------------------------------------
    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()

        # GPU로 배치 텐서 전송
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)

        # Forward Pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # Logits 추출 및 Loss 계산
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs
        loss = criterion(logits, labels)

        # Backward Pass & Gradient Clipping
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Weights & Learning Rate 업데이트
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        if (step + 1) % 50 == 0 or (step + 1) == len(train_dataloader):
            print(f"Batch {step + 1}/{len(train_dataloader)} | Train Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_dataloader)

    # ---------------------------------------------------------
    # [2] Evaluation Loop
    # ---------------------------------------------------------
    model.eval()
    total_val_loss = 0
    correct_preds = 0
    total_preds = 0

    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )

            logits = outputs.logits if hasattr(outputs, 'logits') else outputs
            loss = criterion(logits, labels)
            total_val_loss += loss.item()

            # 정확도(Accuracy) 계산
            preds = torch.argmax(logits, dim=1)
            correct_preds += (preds == labels).sum().item()
            total_preds += labels.size(0)

    avg_val_loss = total_val_loss / len(val_dataloader)
    val_accuracy = correct_preds / total_preds

    print(f"\n[Epoch {epoch + 1} 결과 Summary]")
    print(f"  Average Train Loss: {avg_train_loss:.4f}")
    print(f"  Average Val Loss:   {avg_val_loss:.4f}")
    print(f"  Validation Accuracy: {val_accuracy * 100:.2f}%\n")

model_path = "./custom_bert_model.pt"
torch.save(model.state_dict(), model_path)


======== Epoch 1 / 1 ========
Batch 50/125 | Train Loss: 0.0053
Batch 100/125 | Train Loss: 0.0020
Batch 125/125 | Train Loss: 0.0018

[Epoch 1 결과 Summary]
  Average Train Loss: 0.0831
  Average Val Loss:   0.0013
  Validation Accuracy: 100.00%



학습에 오류가 있다. 왜 그럴까? pooler를 통과한 텐서의 로짓 전체로 손실 계산을 하다보니 오류가 발생한 것이다.